# 01 — Market handles

`tq.MarketEnvironment`: rate curves, discrete dividends, vol surfaces. Q/A: `docs/chapters/rates.tex` (Curves), `equity.tex`.

In [ ]:
import os, sys
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import matplotlib.pyplot as plt
import tensorquant as tq


## Market
Latest synthetic snapshot in `marketdata/<yyyymmdd>/<HHMM>/` (`marketdata/generate_synthetic.py`): 50 tickers (TICKER1-30 EUR, 31-40 GBP, 41-50 USD), one OIS curve per currency, FX, correlations.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
EVALUATION_DATE = tq.Settings.evaluation_date = mkt.evaluation_date

factors = mkt.risk_factors()
kinds = {}
for k in factors:
    kind = f"{k.split(':')[0]}:{k.split(':')[3]}"
    kinds[kind] = kinds.get(kind, 0) + 1
print(f"snapshot {EVALUATION_DATE} / {mkt.calibration_set}:", kinds)

TICKERS = ["TICKER1", "TICKER31", "TICKER41"]   # one per currency

## Rate curves

Key relations: $P(0,T)=e^{-r(T)T}$, $\quad F(0;T_1,T_2)=\frac{1}{\tau}\left(\frac{P(0,T_1)}{P(0,T_2)}-1\right)$, $\quad f(0,T)=r(T)+T\,r'(T)$.

In [ ]:
ir_curves = [(tq.Currency[k.split(":")[1]], k.split(":")[2]) for k in factors if k.startswith("IR:")]

t = np.linspace(0.25, 10.0, 200)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ccy, ticker in ir_curves:
    c = mkt.get_ir_curve(ccy, ticker=ticker)
    df = np.array([float(c.discount(x)) for x in t])
    axes[0].plot(t, -np.log(df) / t * 100, label=ticker)
    axes[1].plot(t, df, label=ticker)
    axes[2].plot(t, [float(c.inst_fwd(x)) * 100 for x in t], label=ticker)
for ax, title in zip(axes, ["Zero rate cc (%)", "Discount factor", "Inst. forward (%)"]):
    ax.set_title(title); ax.set_xlabel("T (Y)"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()

## Discrete dividends

In [ ]:
for ticker in TICKERS:
    ccy = mkt.get_currency(ticker)
    spot = float(mkt.get_eq_spot(ticker, ccy=ccy))
    div = mkt.get_eq_dividends(ticker, ccy=ccy)
    print(f"{ticker} ({ccy.name}): spot={spot:.2f}")
    for ex, amt in list(zip(div.ex_dates, div.amounts))[:6]:
        print(f"   {ex}  {amt:8.4f}  ({amt/spot:.2%})")

## Volatility smiles

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(5 * len(TICKERS), 4))
for ax, ticker in zip(axes, TICKERS):
    ccy = mkt.get_currency(ticker)
    vs = mkt.get_eq_vol_surface(ticker, ccy=ccy)
    spot = float(mkt.get_eq_spot(ticker, ccy=ccy))
    K = np.array(vs.strike)
    for T in vs.maturity:
        ax.plot(K / spot, [float(vs.volatility(strike=x, tenor=T)) * 100 for x in K], "o-", ms=3, label=f"T={T:.2f}")
    ax.set_title(f"{ticker} ({ccy.name}) smile"); ax.set_xlabel("K / spot"); ax.set_ylabel("IV (%)")
    ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
plt.tight_layout()

## Exercises
1. Check numerically $f(0,T) \approx r(T) + T\,r'(T)$ with finite differences.
2. Compute the equity forward $F(0,T)=\big(S_0-\mathrm{PV(div)}\big)/P(0,T)$ for one ticker.